In [ ]:
import os
import time
import threading
import subprocess
import base64
import io
from openai import OpenAI
from PIL import Image, ImageOps
from gpiozero import Button
from escpos.printer import Usb
from signal import pause
from dataclasses import dataclass, field

client = OpenAI(api_key="")

AUDIO_FILENAME = "input.wav"
GPIO_PIN = 17
PRINTER_VID = 0x0485
PRINTER_PID = 0x5741


@dataclass
class Scene:
    animal: str | None = None
    location: str | None = None
    weather: str | None = None
    traits: set[str] = field(default_factory=set)
    actions: set[str] = field(default_factory=set)
    accessories: set[str] = field(default_factory=set)
    exclusions: set[str] = field(default_factory=set)


ANIMALS = ["polar bear", "giraffe", "monkey", "chicken"]
LOCATIONS = ["ocean", "forest", "beach", "city"]
WEATHER_MAP = {
    "snowy": "snowing",
    "snowing": "snowing",
    "rainy": "raining",
    "raining": "raining",
    "sunny": "sunny",
    "cloudy": "cloudy",
}


def detect_scene(prompt: str) -> Scene:
    text = prompt.lower()
    scene = Scene()

    for animal in ANIMALS:
        if animal in text:
            scene.animal = animal

    for location in LOCATIONS:
        if location in text:
            scene.location = location

    for word, weather in WEATHER_MAP.items():
        if word in text:
            scene.weather = weather

    return scene


def apply_rules(scene: Scene) -> Scene:
    animal = scene.animal
    location = scene.location
    weather = scene.weather

    # 🐻‍❄️ Polar bear
    if animal == "polar bear":
        if location == "ocean":
            scene.actions.add("snorkeling")
            scene.exclusions.update({"scarf", "umbrella"})
            scene.accessories.discard("scarf")
            scene.accessories.discard("umbrella")

        if weather == "snowing":
            scene.traits.add("happy")
            if location != "ocean":
                scene.accessories.add("scarf")

        if location == "forest" and weather == "raining":
            scene.accessories.add("umbrella")

        if location == "beach" and weather == "sunny":
            scene.actions.add("getting a suntan")

        if location == "beach" and weather == "cloudy":
            scene.actions.add("building a sandcastle")

        if location == "city":
            scene.traits.add("sculpture")

    # 🦒 Giraffe
    if animal == "giraffe":
        if location in {"ocean", "forest"}:
            scene.traits.add("head out of frame")

        if location == "beach":
            scene.traits.add("with family")

        if location == "beach" and weather == "sunny":
            scene.actions.add("getting a suntan")

        if location == "beach" and weather == "cloudy":
            scene.actions.add("building a sandcastle")

        if location == "beach" and weather == "snowing":
            scene.traits.add("sad")

        if location == "city":
            scene.traits.add("in the zoo")

    # 🐒 Monkey
    if animal == "monkey":
        if location == "ocean":
            scene.actions.add("snorkeling")

        if location == "forest":
            scene.accessories.add("banana")
            if weather == "raining":
                scene.accessories.add("umbrella")

        if location == "beach":
            scene.traits.add("with family")
            if weather == "sunny":
                scene.actions.add("getting a suntan")

        if location == "city":
            scene.traits.add("in the zoo")

        if weather == "sunny":
            scene.accessories.add("shorts")

        if weather in {"raining", "snowing"} and location not in {"ocean", "forest"}:
            scene.traits.add("sad")

    # 🐓 Chicken
    if animal == "chicken":
        if location == "ocean":
            scene.actions.add("snorkeling")

        if weather == "sunny":
            scene.traits.add("fried chicken")

        if weather in {"raining", "snowing"}:
            scene.traits.add("sad")

        if location == "beach" and weather == "sunny":
            scene.actions.add("getting a suntan")

        if location == "beach" and weather == "cloudy":
            scene.actions.add("building a sandcastle")

        if location == "forest" and weather == "raining":
            scene.accessories.add("umbrella")

    return scene


def build_image_prompt(scene: Scene) -> str:
    animal = scene.animal or "animal"

    description = []
    if scene.traits:
        description.extend(sorted(scene.traits))
    description.append(animal)
    if scene.actions:
        description.append(", " + ", ".join(sorted(scene.actions)))
    if scene.location:
        description.append(f"in the {scene.location}")
    if scene.weather:
        description.append(f"on a {scene.weather} day")
    if scene.accessories:
        description.append("with " + ", ".join(sorted(scene.accessories)))
    prompt = "Generate a cartoon image of " + " ".join(description) + "."

    if scene.exclusions:
        prompt += " Do not include " + ", ".join(sorted(scene.exclusions)) + "."

    return prompt

def get_printer():
    try:
        p = Usb(PRINTER_VID, PRINTER_PID)
        p._raw(b'\x1b\x37\x07\xfe\x02')
        p._raw(b'\x12\x23\x80')
        p._raw(b'\x1b\x7b\x01')
        return p
    except Exception as e:
        print(f"Printer Connection Failed: {e}")
        return None

class VoiceRecorder:
    def __init__(self):
        self.process = None
    def start(self):
        print("\n[Recording Started]...")
        cmd = ["arecord", "-D", "hw:3,0", "-f", "S16_LE", "-r", "44100", "-c", "1", AUDIO_FILENAME]
        self.process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    def stop(self):
        if self.process:
            self.process.terminate()
            return True
        return False

recorder = VoiceRecorder()

def process_and_print():
    try:
        with open(AUDIO_FILENAME, "rb") as f:
            transcript = client.audio.transcriptions.create(file=f, model="whisper-1")
        prompt = transcript.text

        print(f"Transcription Result: {prompt}")

        scene = detect_scene(prompt)
        scene = apply_rules(scene)
        prompt = build_image_prompt(scene)

        print(f"Final Image Prompt: {prompt}")


        print("AI Generating Image...")
        result = client.images.generate(
            model="dall-e-3",
            prompt=prompt + " in black-and-white with clear and bold line-based art style and fun",
            size="1024x1024",
            response_format="b64_json"
        )

        image_bytes = base64.b64decode(result.data[0].b64_json)
        img = Image.open(io.BytesIO(image_bytes))

        img = ImageOps.grayscale(img)

        img = img.rotate(180)

        img = img.resize((384, 384), Image.Resampling.LANCZOS)

        img = img.point(lambda x: 0 if x < 127 else 255, '1')

        p = get_printer()
        if p:
            print("Executing Segmented HD Printing...")


            step = 48
            for i in range(0, 384, step):
                box = (0, i, 384, i + step)
                segment = img.crop(box)
                p.image(segment)
                time.sleep(0.3)

            p.text("\n\n\n\n\n")
            p.close()
            print("Printing Finished.")
    except Exception as e:
        print(f"Error: {e}")


btn = Button(GPIO_PIN, pull_up=None, active_state=False, bounce_time=0.2)
btn.when_pressed = recorder.start
btn.when_released = lambda: threading.Thread(target=process_and_print, daemon=True).start() if recorder.stop() else None

print("HD System Ready!")
pause()